# Underwater enhancement: Sea-thru (BlueROV / Moreton Bay)

Sea-thru (Akkaynak and Treibitz, CVPR 2019) removes the water from an image using a **physical model**: it estimates backscatter (the veiling light) and range-dependent attenuation, then inverts both. Unlike CLAHE or FUnIE-GAN, it **needs a per-pixel range map**.

**What this notebook uses**
- Algorithm: vendored from the community port [`hainh/sea-thru`](https://github.com/hainh/sea-thru) (MIT) inside `utils/seathru_utils.py`.
- Range map: **Depth Anything V2 (Small)** monocular depth, rescaled to an assumed metric range.

**Limitations**
- Monocular depth is relative. `min_range_m` and `range_span_m` are guesses about how far the nearest and farthest pixels are. 
- Sea-thru assumes the darkest pixels at each range are shadows (roughly no reflected light). Frames with no real shadows or dark regions over-estimate backscatter.
- The port's default `l=0.5` halves the attenuation correction, so output can look dark or under-corrected.
- For video, each frame is normalised independently, so brightness and colour flicker between frames is expected.
- If real range data exists (vehicle altitude, stereo, sonar), it can be passed as `depth_map=` to skip the monocular guess.

In [ ]:
import sys, time
from pathlib import Path

import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

# Paths (relative to this folder)
VIDEO_PATH = Path("../data/bluerov_videos/Myora-BlueROV.mp4")
OUT_DIR = Path("outputs/seathru")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# seathru_utils.py and coral_utils.py live in the repository's utils folder
sys.path.append("../utils")
import importlib, seathru_utils
importlib.reload(seathru_utils)   # picks up edits without restarting the kernel
from seathru_utils import load_seathru, enhance_frame

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)
print("video exists:", VIDEO_PATH.exists())

## Sample frames from the video

In [ ]:
N_FRAMES = 6

cap = cv2.VideoCapture(str(VIDEO_PATH))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
print(f"{total} frames @ {fps:.1f} fps, {int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}x{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}")

frame_idxs = np.linspace(total * 0.05, total * 0.95, N_FRAMES).astype(int)
frames = []
for idx in frame_idxs:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
    ok, bgr = cap.read()
    if ok:
        frames.append(Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)))
cap.release()
print("sampled", len(frames), "frames at", list(frame_idxs))

fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for ax, im, i in zip(axes.ravel(), frames, frame_idxs):
    ax.imshow(im); ax.set_title(f"frame {i}"); ax.axis("off")
plt.tight_layout(); plt.show()

## Load the depth model (Depth Anything V2 Small, about 25M parameters)

In [ ]:
# If MPS raises an operator error here, set DEPTH_DEVICE = "cpu" (the depth model is small).
DEPTH_DEVICE = device
depth_bundle = load_seathru(DEPTH_DEVICE)
print("depth model loaded on", DEPTH_DEVICE)

## One frame: depth map and Sea-thru result

`min_range_m` and `range_span_m` below are placeholders. They should be set from knowledge of the scene (for example visibility during the dive and typical ROV-to-seabed distance). Runtime is roughly 5 to 10 s per frame at `max_size=640`.

In [ ]:
TEST_IDX = 0
MIN_RANGE_M = 0.5    # assumed range of the nearest pixel (m)
RANGE_SPAN_M = 2.0   # nearest -> farthest span (m)

img = frames[TEST_IDX]
t0 = time.time()
enh, dbg = enhance_frame(img, depth_bundle, DEPTH_DEVICE,
                         min_range_m=MIN_RANGE_M, range_span_m=RANGE_SPAN_M,
                         return_debug=True)
print(f"Sea-thru took {time.time() - t0:.1f}s")

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
axes[0].imshow(img);  axes[0].set_title("Original")
im = axes[1].imshow(dbg["depth"], cmap="turbo"); axes[1].set_title("Assumed range (m)")
plt.colorbar(im, ax=axes[1], fraction=0.046)
axes[2].imshow(dbg["nmap"], cmap="tab20"); axes[2].set_title(f"Depth neighbourhoods ({dbg['nmap'].max()})")
axes[3].imshow(enh);  axes[3].set_title("Sea-thru")
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

print("Per-channel mean (R,G,B)  original:", np.asarray(img).reshape(-1, 3).mean(0).round(1),
      " sea-thru:", np.asarray(enh).reshape(-1, 3).mean(0).round(1))

## Sensitivity to the range assumption

The depth model gives relative depth only, so the output depends on these two numbers. If the result changes drastically across this grid, any downstream segmentation gain or loss should be treated with suspicion.

In [ ]:
MIN_RANGES = [0.3, 1.0]
SPANS = [1.0, 2.0, 4.0]

fig, axes = plt.subplots(len(MIN_RANGES), len(SPANS), figsize=(5 * len(SPANS), 3.6 * len(MIN_RANGES)))
for r, mn in enumerate(MIN_RANGES):
    for c, sp in enumerate(SPANS):
        out = enhance_frame(frames[TEST_IDX], depth_bundle, DEPTH_DEVICE, min_range_m=mn, range_span_m=sp)
        axes[r, c].imshow(out); axes[r, c].axis("off")
        axes[r, c].set_title(f"min={mn} m, span={sp} m")
plt.tight_layout(); plt.show()

## All sampled frames, before and after

In [ ]:
enhanced_frames, times = [], []
for im in frames:
    t0 = time.time()
    enhanced_frames.append(enhance_frame(im, depth_bundle, DEPTH_DEVICE,
                                         min_range_m=MIN_RANGE_M, range_span_m=RANGE_SPAN_M))
    times.append(time.time() - t0)
print(f"mean {np.mean(times):.1f}s/frame")

fig, axes = plt.subplots(len(frames), 2, figsize=(12, 3.3 * len(frames)))
for i, (a, b) in enumerate(zip(frames, enhanced_frames)):
    axes[i, 0].imshow(a); axes[i, 0].set_title(f"original (frame {frame_idxs[i]})")
    axes[i, 1].imshow(b); axes[i, 1].set_title("Sea-thru")
    axes[i, 0].axis("off"); axes[i, 1].axis("off")
plt.tight_layout(); plt.show()

## Effect on Coralscapes segmentation

There is no ground truth for these frames, so this does **not** measure accuracy. It shows how much the predicted segmentation *changes* after enhancement and how class coverage shifts. The controlled test of whether enhancement helps, scored against expert annotations, is `zeroshot/coral_enhancement_comparison.ipynb`.

In [ ]:
from coral_utils import load_coralscapes_model, segment_image, colourise_mask, coverage_stats

preprocessor, seg_model = load_coralscapes_model(device)

def agreement(a, b):
    """Pixel agreement and mean IoU between two predicted label maps (NOT accuracy)."""
    pix = float((a == b).mean())
    ious = []
    for c in np.union1d(np.unique(a), np.unique(b)):
        inter = np.logical_and(a == c, b == c).sum()
        union = np.logical_or(a == c, b == c).sum()
        if union: ious.append(inter / union)
    return pix, float(np.mean(ious))

seg_rows = []
for i, (raw, enh_i) in enumerate(zip(frames, enhanced_frames)):
    p_raw = segment_image(raw, preprocessor, seg_model, device)
    p_enh = segment_image(enh_i, preprocessor, seg_model, device)
    if i == 0:
        print("predicted id range (raw):", p_raw.min(), "-", p_raw.max())  # resolves the 0-39 vs 0-38 question
    seg_rows.append((raw, enh_i, p_raw, p_enh))
    pix, miou = agreement(p_raw, p_enh)
    print(f"frame {frame_idxs[i]}: pixel agreement raw vs sea-thru = {pix:.3f}, mean IoU between predictions = {miou:.3f}")

In [ ]:
fig, axes = plt.subplots(len(seg_rows), 4, figsize=(20, 3.3 * len(seg_rows)))
for i, (raw, enh_i, p_raw, p_enh) in enumerate(seg_rows):
    for ax, im, t in zip(axes[i], [raw, colourise_mask(p_raw), enh_i, colourise_mask(p_enh)],
                         ["original", "segmentation (original)", "Sea-thru", "segmentation (Sea-thru)"]):
        ax.imshow(im); ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.show()

# top-5 class coverage, first frame
print("raw:", [(n, round(p, 1)) for n, p in coverage_stats(seg_rows[0][2])[:5]])
print("sea-thru:", [(n, round(p, 1)) for n, p in coverage_stats(seg_rows[0][3])[:5]])

## Save results

In [ ]:
for i, (raw, enh_i) in enumerate(zip(frames, enhanced_frames)):
    raw.save(OUT_DIR / f"frame{frame_idxs[i]:06d}_original.png")
    enh_i.save(OUT_DIR / f"frame{frame_idxs[i]:06d}_seathru.png")
print("saved to", OUT_DIR)